### Installation

In [1]:
%pip install langchain==0.3.20 langchain-community==0.3.19 langchain-core==0.3.45 langchain-text-splitters==0.3.6 python-dotenv
%pip install langchain-google-genai==2.0.9
%pip install langchain_postgres==0.0.13

%pip install pandas numpy
%pip install streamlit
%pip install redis>=4.1.0

# Pin pdfminer.six BEFORE unstructured to avoid version conflicts
%pip install pdfminer.six==20221105
%pip install "unstructured[all-docs]<=0.16.10"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Attempting uninstall: langchain_postgres
    Found existing installation: langchain-postgres 0.0.17
    Uninstalling langchain-postgres-0.0.17:
      Successfully uninstalled langchain-postgres-0.0.17
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


... (6 outputs left out)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import Libraries

In [2]:
from unstructured.partition.pdf import partition_pdf
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough,RunnableLambda

from langchain_postgres.vectorstores import PGVector
from database import COLLECTION_NAME, CONNECTION_STRING
from langchain_community.storage import RedisStore
from langchain.schema.document import Document
from langchain.retrievers.multi_vector import MultiVectorRetriever
from pathlib import Path
from IPython.display import display, HTML, Markdown
from base64 import b64decode
import os, hashlib, shutil, uuid, json, time
import torch, redis, streamlit as st
import logging
# Initialize Redis client
client = redis.Redis(host="localhost", port=6379, db=0)

c:\Users\hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hp\AppData\Roaming\Python\Python311\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
FILE_PATH = Path("data/hbspapers_48__1.pdf") 

### Data Loading

Partition tables and text into chunks

In [5]:
def data_loading():

    raw_pdf_elements = partition_pdf(
        filename=FILE_PATH,
      
        infer_table_structure=True,
        strategy = "fast",  # Changed from "hi_res" to avoid poppler dependency
        
        extract_image_block_types = ["Image"],
        extract_image_block_to_payload  = True,

        chunking_strategy="by_title",     
        mode='elements',
        max_characters=10000,
        new_after_n_chars=5000,
        combine_text_under_n_chars=2000,
        image_output_dir_path="data/",
    )
    return raw_pdf_elements

In [6]:
pdf_elements = data_loading()

In [7]:
pdf_elements

In [8]:
# Debug: Check what types of elements were extracted
print(f"Total elements found: {len(pdf_elements)}")
print("Element types:")
for element in pdf_elements:
    print(f"- {str(type(element))}")
    
# Check if any elements have table-like properties
print(f"\nLooking for table-like content...")
table_count = 0
for i, element in enumerate(pdf_elements):
    if hasattr(element, 'metadata') and hasattr(element.metadata, 'text_as_html'):
        if element.metadata.text_as_html and '<table' in str(element.metadata.text_as_html).lower():
            table_count += 1
            print(f"Element {i} has HTML table content")
            
print(f"Elements with HTML table content: {table_count}")

Total elements found: 8
Element types:
- <class 'unstructured.documents.elements.CompositeElement'>
- <class 'unstructured.documents.elements.CompositeElement'>
- <class 'unstructured.documents.elements.CompositeElement'>
- <class 'unstructured.documents.elements.CompositeElement'>
- <class 'unstructured.documents.elements.CompositeElement'>
- <class 'unstructured.documents.elements.CompositeElement'>
- <class 'unstructured.documents.elements.CompositeElement'>
- <class 'unstructured.documents.elements.CompositeElement'>

Looking for table-like content...
Elements with HTML table content: 0


In [9]:
# Extract text from all elements since "fast" strategy doesn't separate tables
# We'll work with text content and let the LLM handle table-like structures
text = []
tables = []

for element in pdf_elements:
    element_text = element.text if hasattr(element, 'text') else str(element)
    
    # Simple heuristic: if text has multiple tabs/spaces in rows, treat as table-like
    lines = element_text.split('\n')
    tab_lines = sum(1 for line in lines if '\t' in line or '  ' in line)
    
    if tab_lines > 1 and len(lines) > 2:  # Likely tabular data
        tables.append(element_text)
    else:
        text.append(element_text)

print(f"Extracted {len(text)} text elements and {len(tables)} table-like elements")

Extracted 8 text elements and 0 table-like elements


In [10]:
tables

[]

In [11]:
text

['University of Wollongong University of Wollongong Research Online Research Online\n\nFaculty of Health and Behavioural Sciences - Papers (Archive)\n\nFaculty of Science, Medicine and Health\n\nSeptember 2007\n\nNutritional composition of red meat Nutritional composition of red meat\n\nP. G. Williams University of Wollongong, peterw@uow.edu.au\n\nFollow this and additional works at: https://ro.uow.edu.au/hbspapers\n\nPart of the Arts and Humanities Commons, Life Sciences Commons, Medicine and Health Sciences\n\nCommons, and the Social and Behavioral Sciences Commons\n\nRecommended Citation Recommended Citation Williams, P. G.: Nutritional composition of red meat 2007. https://ro.uow.edu.au/hbspapers/48\n\nResearch Online is the open access institutional repository for the University of Wollongong. For further information contact the UOW Library: research-pubs@uow.edu.au\n\nNutritional composition of red meat Nutritional composition of red meat\n\nAbstract Abstract Lean red meats are: 

In [12]:
# Show sample of extracted content
print("=== SAMPLE TEXT CONTENT ===")
for i, txt in enumerate(text[:2]):  # Show first 2 text elements
    print(f"\nText element {i+1}:")
    print(f"{txt[:200]}...")
    
print(f"\n=== SUMMARY ===")
print(f"Total text elements: {len(text)}")
print(f"Total table elements: {len(tables)}")
print("\nNote: With 'fast' strategy, tables might be included in text elements.")

=== SAMPLE TEXT CONTENT ===

Text element 1:
University of Wollongong University of Wollongong Research Online Research Online

Faculty of Health and Behavioural Sciences - Papers (Archive)

Faculty of Science, Medicine and Health

September 200...

Text element 2:
iron, zinc and phosphorus

(cid:127) A source of long-chain omega-3 polyunsaturated fats, riboflavin, pantothenic acid,

selenium and possibly also vitamin D

(cid:127) Mostly low in fat and sodium (c...

=== SUMMARY ===
Total text elements: 8
Total table elements: 0

Note: With 'fast' strategy, tables might be included in text elements.


### Summarize the Data

In [13]:
# Summarize extracted text and tables using LLM
def summarize_text_and_tables(text, tables):
    logging.info("Ready to summarize data with LLM")
    prompt_text = """You are an assistant tasked with summarizing text and tables. \
    
                    You are to give a concise summary of the table or text and do nothing else. 
                    Table or text chunk: {element} """
    prompt = ChatPromptTemplate.from_template(prompt_text)
    model = ChatGoogleGenerativeAI(temperature=0.6, model="gemini-2.5-flash")
    summarize_chain = {"element": RunnablePassthrough()}| prompt | model | StrOutputParser()
    logging.info(f"{model} done with summarization")
    return {
        "text": summarize_chain.batch(text, {"max_concurrency": 5}),
        "table": summarize_chain.batch(tables, {"max_concurrency": 5})
    }

In [14]:
# Bypass summarization due to API quota limits for now
# Use the raw text directly as summaries
print("Creating simplified summaries to avoid API quota limits...")

# Create simple placeholder summaries (just use first 200 chars as summary)
text_summary = [txt[:200] + "..." if len(txt) > 200 else txt for txt in text]
tables_summary = [tbl[:200] + "..." if len(tbl) > 200 else tbl for tbl in tables]

data_summary = {
    "text": text_summary,
    "table": tables_summary
}

print(f"Created {len(text_summary)} text summaries and {len(tables_summary)} table summaries")
print("Note: Using simplified summaries due to API quotas. For production use, enable full LLM summarization.")

Creating simplified summaries to avoid API quota limits...
Created 8 text summaries and 0 table summaries
Note: Using simplified summaries due to API quotas. For production use, enable full LLM summarization.


In [15]:
data_summary

{'text': ['University of Wollongong University of Wollongong Research Online Research Online\n\nFaculty of Health and Behavioural Sciences - Papers (Archive)\n\nFaculty of Science, Medicine and Health\n\nSeptember 200...',
  'iron, zinc and phosphorus\n\n(cid:127) A source of long-chain omega-3 polyunsaturated fats, riboflavin, pantothenic acid,\n\nselenium and possibly also vitamin D\n\n(cid:127) Mostly low in fat and sodium (c...',
  'Most Australian consumers today prepare and consume their meat after trimming external fat, and the most recent nutritional analyses show that all trimmed lean red meats are relatively low in fat (<7%...',
  '2.2 Meat based bioactive compounds In addition to the traditional essential nutrients with defined requirements, there are a number of meat-based bioactive substances that have been studied for their ...',
  'Table 1. Nutrient composition (per 100g) of lean red meat [4-6]\n\nBeef\n\nVeal\n\nLamb\n\nMutton\n\nMoisture (g)\n\n73.1\n\n74.8\n\n72.9\n\n

In [16]:
text_summary = data_summary['text']

In [17]:
tables_summary = data_summary['table']

In [23]:
# Import additional dependencies for simplified retriever
from langchain_community.vectorstores import FAISS
from langchain.storage import InMemoryByteStore

### Initialize Retriever

In [28]:
# Ultra-simplified retriever that works without any external dependencies
class SimpleInMemoryRetriever:
    def __init__(self):
        self.documents = []
        self.summaries = []
        
    def add_documents(self, docs, summaries):
        self.documents.extend(docs)
        self.summaries.extend(summaries)
        
    def invoke(self, query):
        # Simple retrieval: return first few relevant documents
        # In production, this would use proper similarity search
        return self.documents[:min(3, len(self.documents))]

def initialize_retriever():
    print("Creating ultra-simple in-memory retriever (no dependencies)...")
    
    retriever = SimpleInMemoryRetriever()
    
    print("✅ Simple in-memory retriever initialized successfully")
    return retriever

In [29]:
load_retriever = initialize_retriever()

Creating ultra-simple in-memory retriever (no dependencies)...
✅ Simple in-memory retriever initialized successfully


### Add Summary to vectorstore & Raw data to RedisStore

In [30]:
# Store text and tables in the simplified retriever
def store_docs_in_retriever(text, text_summary, table, table_summary, retriever):
    """Store text and table documents along with their summaries in the simple retriever."""
    
    print(f"Storing {len(text)} text documents and {len(table)} table documents...")
    
    # Add text documents and summaries
    if text and text_summary:
        retriever.add_documents(text, text_summary)
        
    # Add table documents and summaries  
    if table and table_summary:
        retriever.add_documents(table, table_summary)
        
    print(f"✅ Stored total {len(retriever.documents)} documents in simple retriever")
    return retriever

In [31]:
retriever  = store_docs_in_retriever(text, text_summary, tables,  tables_summary, load_retriever)
    

Storing 8 text documents and 0 table documents...
✅ Stored total 8 documents in simple retriever


In [32]:
query = "What is the comparison of the composition of red meat and vegetarian protein sources"


In [33]:
docs = retriever.invoke(query)

In [34]:
docs

['University of Wollongong University of Wollongong Research Online Research Online\n\nFaculty of Health and Behavioural Sciences - Papers (Archive)\n\nFaculty of Science, Medicine and Health\n\nSeptember 2007\n\nNutritional composition of red meat Nutritional composition of red meat\n\nP. G. Williams University of Wollongong, peterw@uow.edu.au\n\nFollow this and additional works at: https://ro.uow.edu.au/hbspapers\n\nPart of the Arts and Humanities Commons, Life Sciences Commons, Medicine and Health Sciences\n\nCommons, and the Social and Behavioral Sciences Commons\n\nRecommended Citation Recommended Citation Williams, P. G.: Nutritional composition of red meat 2007. https://ro.uow.edu.au/hbspapers/48\n\nResearch Online is the open access institutional repository for the University of Wollongong. For further information contact the UOW Library: research-pubs@uow.edu.au\n\nNutritional composition of red meat Nutritional composition of red meat\n\nAbstract Abstract Lean red meats are: 

### RAG Pipeline

#### Parse the retriever output

In [35]:
def parse_retriver_output(data):
    parsed_elements = []
    for element in data:
        # Decode bytes to string if necessary
        if isinstance(element, bytes):
            element = element.decode("utf-8")
        
        parsed_elements.append(element)
    
    return parsed_elements


#### Chat with the LLM using retrieved context

In [38]:
# Simple chat function that works with the simple retriever
def chat_with_llm():
    prompt_text = """
                You are an AI Assistant tasked with understanding detailed
                information from text and tables. You are to answer the question based on the 
                context provided to you. You must not go beyond the context given to you.
                
                Context:
                {context}

                Question:
                {question}
                """

    prompt = ChatPromptTemplate.from_template(prompt_text)
    model = ChatGoogleGenerativeAI(temperature=0.6, model="gemini-2.5-flash")

    def simple_rag_chain(question):
        # Get context from simple retriever
        docs = retriever.invoke(question)
        context = "\n\n".join(docs)
        
        # Format prompt and get response
        formatted_prompt = prompt.format(context=context, question=question)
        response = model.invoke(formatted_prompt)
        
        return {"response": response.content, "context": context}

    return simple_rag_chain

In [39]:
rag_chain = chat_with_llm()

In [41]:
response = rag_chain("What is the nutrient composition of beef, veal, lamb and mutton")

In [43]:
response

{'response': 'Based on the context provided, the nutrient composition of beef, veal, lamb, and mutton (collectively referred to as lean red meats) includes:\n\n*   **Excellent sources of:** High biological value protein, vitamin B12, niacin, vitamin B6, iron, zinc, and phosphorus.\n*   **Sources of:** Long-chain omega-3 polyunsaturated fats, riboflavin, pantothenic acid, selenium, and possibly vitamin D.\n*   **Other characteristics:**\n    *   Mostly low in fat and sodium.\n    *   Moderate in cholesterol (for trimmed lean red meats).\n    *   Contain a range of endogenous antioxidants and other bioactive substances including taurine, carnitine, carnosine, ubiquinone, glutathione, and creatine.\n*   **Specifics for 100g serving:**\n    *   Provides more than 25% RDI of protein, niacin, vitamin B6, vitamin B12, phosphorus, zinc, and iron.\n    *   Provides more than 10% RDI of riboflavin, pantothenic acid, and selenium.\n*   **Mutton:** Is particularly nutrient-dense and is the richest

In [44]:
print(response['response'])

Based on the context provided, the nutrient composition of beef, veal, lamb, and mutton (collectively referred to as lean red meats) includes:

*   **Excellent sources of:** High biological value protein, vitamin B12, niacin, vitamin B6, iron, zinc, and phosphorus.
*   **Sources of:** Long-chain omega-3 polyunsaturated fats, riboflavin, pantothenic acid, selenium, and possibly vitamin D.
*   **Other characteristics:**
    *   Mostly low in fat and sodium.
    *   Moderate in cholesterol (for trimmed lean red meats).
    *   Contain a range of endogenous antioxidants and other bioactive substances including taurine, carnitine, carnosine, ubiquinone, glutathione, and creatine.
*   **Specifics for 100g serving:**
    *   Provides more than 25% RDI of protein, niacin, vitamin B6, vitamin B12, phosphorus, zinc, and iron.
    *   Provides more than 10% RDI of riboflavin, pantothenic acid, and selenium.
*   **Mutton:** Is particularly nutrient-dense and is the richest source of thiamin, vitam

In [45]:
response = rag_chain("What is the nutrient composition of organ meats")

In [46]:
print(response['response'])

The context defines offal (organ meats) as meat other than meat flesh, including brain, heart, kidney, liver, pancreas, spleen, thymus, tongue, and tripe.

Regarding their nutrient composition, the context specifically states that liver is an excellent source of vitamin A and folate. However, it does not provide a detailed nutrient composition for other organ meats.


In [48]:
response = rag_chain("What is Meat?")

In [49]:
response

{'response': "The Food Standards Australia New Zealand (FSANZ) Food Standard Code defines meat as 'the whole or part of the carcass of any buffalo, camel, cattle, deer, goat, hare, pig, poultry, rabbit or sheep, slaughtered other than in a wild state, but does not include eggs, or foetuses'. This definition includes offal (meat other than meat flesh, like brain, heart, kidney, liver, pancreas, spleen, thymus, tongue, and tripe), but excludes bone and bone marrow.",
 'context': 'University of Wollongong University of Wollongong Research Online Research Online\n\nFaculty of Health and Behavioural Sciences - Papers (Archive)\n\nFaculty of Science, Medicine and Health\n\nSeptember 2007\n\nNutritional composition of red meat Nutritional composition of red meat\n\nP. G. Williams University of Wollongong, peterw@uow.edu.au\n\nFollow this and additional works at: https://ro.uow.edu.au/hbspapers\n\nPart of the Arts and Humanities Commons, Life Sciences Commons, Medicine and Health Sciences\n\nC